# Excel vs Lake Q1 Intersections

Сверка пересечений уникальных `agr_id` и `ИНН` между Excel-отчетами за январь/февраль/март и `ods_alpha.scd1_agreements`.
В Lake используется month-active SA-срез.

In [ ]:
import pandas as pd
from rail_connectors.connection import connect

month_sources = [
    {
        'report_month': '2026-01-01',
        'excel_path': '/home/jovyan/documents/Equaring/Data/01_Январь_2026.xlsx',
        'header': 1,
    },
    {
        'report_month': '2026-02-01',
        'excel_path': '/home/jovyan/documents/Equaring/Data/02_Февраль_2026.xlsx',
        'header': 0,
    },
    {
        'report_month': '2026-03-01',
        'excel_path': '/home/jovyan/documents/Equaring/Data/03_Март_2026.xlsx',
        'header': 0,
    },
]

AGR_COL_CANDIDATES = ['ID договора', 'agr_id', 'abs_agr_id']
INN_COL_CANDIDATES = ['ИНН', 'inn', 'c_inn']


In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()
print('Impala connection initialized')


In [ ]:
def pick_col(columns, candidates):
    cols = list(columns)
    lower_map = {str(c).strip().lower(): c for c in cols}
    for cand in candidates:
        if cand in cols:
            return cand
        key = str(cand).strip().lower()
        if key in lower_map:
            return lower_map[key]
    return None


def norm_agr(s: pd.Series) -> pd.Series:
    return (
        s.astype(str)
        .str.strip()
        .str.replace(r'\\.0$', '', regex=True)
        .replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA})
    )


def norm_inn(s: pd.Series) -> pd.Series:
    x = (
        s.astype(str)
        .str.strip()
        .str.replace(r'\\.0$', '', regex=True)
        .str.replace(r'\\D', '', regex=True)
        .replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA})
    )

    def _fix_len(v):
        if pd.isna(v):
            return pd.NA
        n = len(v)
        if n == 9:
            return v.zfill(10)
        if n == 11:
            return v.zfill(12)
        if n in (10, 12):
            return v
        return pd.NA

    return x.map(_fix_len)


In [ ]:
summary_rows = []
only_lake_inn_by_month = {}
lake_rows_top5_by_month = {}
reason_rows = []
terms_closed_sample_by_month = {}

for src in month_sources:
    report_month = src['report_month']
    report_ts = pd.to_datetime(report_month)
    month_start = report_ts.strftime('%Y-%m-%d')
    month_end = (report_ts + pd.offsets.MonthEnd(1)).strftime('%Y-%m-%d')
    month_start_dt = pd.to_datetime(month_start)
    month_end_dt = pd.to_datetime(month_end)

    excel_raw = pd.read_excel(src['excel_path'], header=src.get('header', 0))
    agr_col = pick_col(excel_raw.columns, AGR_COL_CANDIDATES)
    inn_col = pick_col(excel_raw.columns, INN_COL_CANDIDATES)

    if agr_col is None or inn_col is None:
        raise ValueError(
            f"[{report_month}] Не найдены колонки agr_id/ИНН. Доступные: {list(excel_raw.columns)}"
        )

    excel_df = pd.DataFrame({
        'agr_id_norm': norm_agr(excel_raw[agr_col]),
        'inn_norm': norm_inn(excel_raw[inn_col]),
    })

    with imp:
        lake_full = imp.fetch(f"""
            select
                a.*,
                c.*,
                cast(a.abs_agr_id as string) as agr_id,
                cast(a.n_agr as string) as n_agr_str,
                cast(a.d_valid_from as date) as agr_d_valid_from,
                cast(a.d_valid_to as date) as agr_d_valid_to,
                regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn
            from ods_alpha.scd1_agreements a
            join ods_alpha.scd1_companies c
              on c.n_cmp = a.n_cmp_client
            where upper(trim(cast(a.acq_class as string))) = 'SA'
              and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
              and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
              and coalesce(a.ods_deleted_flg, '0') <> '1'
              and coalesce(c.ods_deleted_flg, '0') <> '1'
              and c.c_inn is not null
        """)

    if lake_full is None:
        lake_full = pd.DataFrame(columns=['agr_id', 'inn', 'n_agr_str', 'agr_d_valid_from', 'agr_d_valid_to'])

    lake_full = lake_full.copy()
    lake_full['agr_id_norm'] = norm_agr(lake_full['agr_id'])
    lake_full['inn_norm'] = norm_inn(lake_full['inn'])

    set_excel_agr = set(excel_df['agr_id_norm'].dropna().drop_duplicates().tolist())
    set_excel_inn = set(excel_df['inn_norm'].dropna().drop_duplicates().tolist())
    set_lake_agr = set(lake_full['agr_id_norm'].dropna().drop_duplicates().tolist())
    set_lake_inn = set(lake_full['inn_norm'].dropna().drop_duplicates().tolist())

    agr_intersect = set_excel_agr & set_lake_agr
    agr_only_excel = set_excel_agr - set_lake_agr
    agr_only_lake = set_lake_agr - set_excel_agr

    inn_intersect = set_excel_inn & set_lake_inn
    inn_only_excel = set_excel_inn - set_lake_inn
    inn_only_lake = set_lake_inn - set_excel_inn

    only_lake_inn_by_month[report_month] = sorted(list(inn_only_lake))

    lake_rows_top5_by_month[report_month] = (
        lake_full[
            lake_full['inn_norm'].isin(inn_only_lake)
        ]
        .drop_duplicates(subset=['inn_norm'])
        .head(5)
        .copy()
    )

    # ---- Новая проверка: AGR_TERMS ----
    # Ищем кейсы, где для only_lake ИНН в agr_terms нет активной записи в месяце,
    # а последняя запись закрыта до month_start.
    only_lake_scope = (
        lake_full[
            lake_full['inn_norm'].isin(inn_only_lake)
        ][['inn_norm', 'agr_id_norm', 'n_agr_str', 'agr_d_valid_from', 'agr_d_valid_to']]
        .dropna(subset=['inn_norm', 'n_agr_str'])
        .drop_duplicates()
        .copy()
    )

    terms_flags = pd.DataFrame(columns=[
        'n_agr_str',
        'has_active_term_in_month',
        'latest_term_closed_before_month_start',
        'term_d_valid_from_latest',
        'term_d_valid_to_latest',
    ])

    if len(only_lake_scope) > 0:
        n_agr_values = sorted(only_lake_scope['n_agr_str'].astype(str).dropna().unique().tolist())
        if n_agr_values:
            n_agr_sql = ', '.join([f"'{str(x).replace("'", "''")}'" for x in n_agr_values])

            with imp:
                terms_df = imp.fetch(f"""
                    select
                        t.*,
                        cast(t.n_agr as string) as n_agr_str,
                        cast(t.d_valid_from as date) as term_d_valid_from,
                        cast(t.d_valid_to as date) as term_d_valid_to
                    from ods_alpha.scd1_agr_terms t
                    where cast(t.n_agr as string) in ({n_agr_sql})
                      and coalesce(t.ods_deleted_flg, '0') <> '1'
                """)

            if terms_df is None:
                terms_df = pd.DataFrame(columns=['n_agr_str', 'term_d_valid_from', 'term_d_valid_to'])

            if len(terms_df) > 0:
                terms_df = terms_df.copy()
                terms_df['term_d_valid_from_dt'] = pd.to_datetime(terms_df['term_d_valid_from'], errors='coerce')
                terms_df['term_d_valid_to_dt'] = pd.to_datetime(terms_df['term_d_valid_to'], errors='coerce')

                if 'ods_insert_ts' in terms_df.columns:
                    terms_df['ods_insert_ts_dt'] = pd.to_datetime(terms_df['ods_insert_ts'], errors='coerce')
                else:
                    terms_df['ods_insert_ts_dt'] = pd.NaT

                terms_df['is_active_in_month'] = (
                    (terms_df['term_d_valid_from_dt'].notna()) &
                    (terms_df['term_d_valid_from_dt'] <= month_end_dt) &
                    (
                        terms_df['term_d_valid_to_dt'].isna() |
                        (terms_df['term_d_valid_to_dt'] >= month_start_dt)
                    )
                ).astype(int)

                active_by_n_agr = (
                    terms_df
                    .groupby('n_agr_str', as_index=False)['is_active_in_month']
                    .max()
                    .rename(columns={'is_active_in_month': 'has_active_term_in_month'})
                )

                latest_terms = (
                    terms_df
                    .sort_values(['n_agr_str', 'term_d_valid_from_dt', 'ods_insert_ts_dt'], ascending=[True, False, False])
                    .drop_duplicates(subset=['n_agr_str'], keep='first')
                    [['n_agr_str', 'term_d_valid_from', 'term_d_valid_to', 'term_d_valid_to_dt']]
                    .rename(columns={
                        'term_d_valid_from': 'term_d_valid_from_latest',
                        'term_d_valid_to': 'term_d_valid_to_latest'
                    })
                )

                latest_terms['latest_term_closed_before_month_start'] = (
                    latest_terms['term_d_valid_to_dt'].notna() &
                    (latest_terms['term_d_valid_to_dt'] < month_start_dt)
                ).astype(int)

                latest_terms = latest_terms.drop(columns=['term_d_valid_to_dt'])

                terms_flags = (
                    latest_terms
                    .merge(active_by_n_agr, on='n_agr_str', how='outer')
                )

    only_lake_diag = only_lake_scope.merge(terms_flags, on='n_agr_str', how='left')
    if 'has_active_term_in_month' not in only_lake_diag.columns:
        only_lake_diag['has_active_term_in_month'] = 0
    if 'latest_term_closed_before_month_start' not in only_lake_diag.columns:
        only_lake_diag['latest_term_closed_before_month_start'] = 0

    only_lake_diag['has_active_term_in_month'] = only_lake_diag['has_active_term_in_month'].fillna(0).astype(int)
    only_lake_diag['latest_term_closed_before_month_start'] = only_lake_diag['latest_term_closed_before_month_start'].fillna(0).astype(int)

    # Новый этап: после расчета only_lake применяем фильтры agr_terms (P + RSHB + active-in-month)
    terms_prshb_flags = pd.DataFrame(columns=['n_agr_str', 'has_prshb_active_term'])
    if len(only_lake_scope) > 0:
        n_agr_values_for_prshb = sorted(only_lake_scope['n_agr_str'].astype(str).dropna().unique().tolist())
        if n_agr_values_for_prshb:
            n_agr_prshb_sql = ', '.join([f"'{str(x).replace("'", "''")}'" for x in n_agr_values_for_prshb])
            with imp:
                prshb_terms = imp.fetch(f"""
                    select distinct
                        cast(t.n_agr as string) as n_agr_str
                    from ods_alpha.scd1_agr_terms t
                    where cast(t.n_agr as string) in ({n_agr_prshb_sql})
                      and upper(trim(cast(t.cf_ter_type as string))) = 'P'
                      and upper(trim(cast(t.c_fiid_grp as string))) = 'RSHB'
                      and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
                      and (t.d_valid_to is null or cast(t.d_valid_to as date) >= cast('{month_start}' as date))
                      and coalesce(t.ods_deleted_flg, '0') <> '1'
                """)
            if prshb_terms is None:
                prshb_terms = pd.DataFrame(columns=['n_agr_str'])
            if len(prshb_terms) > 0:
                terms_prshb_flags = prshb_terms.copy()
                terms_prshb_flags['has_prshb_active_term'] = 1

    only_lake_diag = only_lake_diag.merge(terms_prshb_flags, on='n_agr_str', how='left')
    only_lake_diag['has_prshb_active_term'] = only_lake_diag['has_prshb_active_term'].fillna(0).astype(int)

    terms_closed_mask = (
        (only_lake_diag['has_active_term_in_month'] == 0) &
        (only_lake_diag['latest_term_closed_before_month_start'] == 1)
    )

    inn_terms_closed_cnt = int(only_lake_diag.loc[terms_closed_mask, 'inn_norm'].nunique())
    inn_only_lake_total = len(only_lake_inn_by_month[report_month])

    # Сколько only_lake ИНН сохраняется после фильтра P/RSHB/active-in-month
    inn_only_lake_after_prshb = int(
        only_lake_diag.loc[only_lake_diag['has_prshb_active_term'] == 1, 'inn_norm']
        .nunique()
    )

    inn_other_cnt = max(inn_only_lake_total - inn_terms_closed_cnt, 0)

    reason_rows.append({
        'month': report_month,
        'reason': 'agr_terms_closed_before_month_start',
        'inn_cnt': inn_terms_closed_cnt,
        'share_pct_of_only_lake': round(100.0 * inn_terms_closed_cnt / inn_only_lake_total, 2) if inn_only_lake_total else 0.0,
    })
    reason_rows.append({
        'month': report_month,
        'reason': 'only_lake_after_prshb_active_filter',
        'inn_cnt': inn_only_lake_after_prshb,
        'share_pct_of_only_lake': round(100.0 * inn_only_lake_after_prshb / inn_only_lake_total, 2) if inn_only_lake_total else 0.0,
    })
    reason_rows.append({
        'month': report_month,
        'reason': 'other_only_lake_reasons',
        'inn_cnt': inn_other_cnt,
        'share_pct_of_only_lake': round(100.0 * inn_other_cnt / inn_only_lake_total, 2) if inn_only_lake_total else 0.0,
    })

    terms_closed_sample_by_month[report_month] = (
        only_lake_diag[terms_closed_mask]
        [['inn_norm', 'n_agr_str', 'agr_id_norm', 'agr_d_valid_from', 'agr_d_valid_to',
          'term_d_valid_from_latest', 'term_d_valid_to_latest',
          'has_active_term_in_month', 'latest_term_closed_before_month_start', 'has_prshb_active_term']]
        .drop_duplicates()
        .head(50)
        .copy()
    )

    summary_rows.append({
        'report_month': report_month,
        'agr_intersect_cnt': len(agr_intersect),
        'inn_intersect_cnt': len(inn_intersect),
    })

summary = pd.DataFrame(summary_rows).sort_values('report_month').reset_index(drop=True)
reason_summary = pd.DataFrame(reason_rows).sort_values(['month', 'reason']).reset_index(drop=True)

print('=== Пересечения Excel vs Lake по месяцам ===')
display(summary)

for month_key in summary['report_month'].tolist():
    print(f"\n===== {month_key} =====")
    print(f"ИНН только в озере: {len(only_lake_inn_by_month[month_key])}")
    print('Примеры 5 строк (полные атрибуты) для ИНН только в озере:')
    display(lake_rows_top5_by_month[month_key])

print('\n=== reason_summary (only_lake) ===')
display(reason_summary)

for month_key in summary['report_month'].tolist():
    print(f"\n===== {month_key}: sample agr_terms_closed_before_month_start =====")
    display(terms_closed_sample_by_month[month_key])
